# M2 · E3 Pelatihan dengan tile dari citra native 1280

Apakah pelatihan dengan potongan citra native meningkatkan deteksi objek kecil? Dua YOLO26n mulai dari bobot resmi yang sama, memakai sumber train yang sama dan jumlah langkah optimizer yang sama pada input model 640. Kontrol melihat citra utuh yang diskalakan menjadi 640. Perlakuan melihat tile 640 dari citra native 1280 dengan overlap nominal 20% dan visibilitas minimal 50%. Tile kosong dibatasi paling banyak satu per tiga tile berobjek. Kontrol mengulang citra yang sama secara seimbang sampai anggaran sampelnya sama dengan perlakuan.

Kedua model dinilai pada 356 gambar validasi utuh berukuran 1280 (E3a), lalu model tile dinilai lagi dengan SAHI dari bobot yang sama (E3b). Observasi terkunci pada split validasi. Pilot historis ubin 384 dari data olahan 640 merupakan percobaan berbeda.

In [ ]:
import os
import subprocess
import sys

os.environ["YOLO_CONFIG_DIR"] = "/tmp/ciringoni-yolo"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.210", "pycocotools==2.0.11", "sahi==0.14.0"],
    check=True,
)

In [ ]:
from hashlib import file_digest, sha256 as digest
import json
from pathlib import Path
import shutil
from tempfile import TemporaryDirectory
from urllib.request import urlretrieve
from zipfile import ZipFile

from IPython.display import display, Image as DisplayImage
from PIL import Image, ImageDraw
import torch

SOURCE_COMMIT = "ee8c14bd132a07db3dae4bb15de0d8a22f517d60"
WEIGHTS_SHA256 = "9b09cc8bf347f0fc8a5f7657480587f25db09b34bf33b0652110fb03a8ad4fef"

def sha256(path):
    with path.open("rb") as stream:
        return file_digest(stream, "sha256").hexdigest()

import os
import platform
cpus = sorted(os.sched_getaffinity(0))[:4]
assert len(cpus) == 4
os.sched_setaffinity(0, cpus)
os.environ.update(OMP_NUM_THREADS="4", MKL_NUM_THREADS="4")
torch.set_num_threads(4)
print("Setting up four-thread Kaggle CPU training", flush=True)
input_root = Path("/kaggle/input")
scratch = TemporaryDirectory(prefix="ciringoni-e3-")
work = Path(scratch.name)
native = work / "native"
project = work / "Data_Mining_Ciringoni_M1"
output = Path("/kaggle/working/E3")
output.mkdir(parents=True)
archives = sorted(input_root.rglob("native_train_val.zip"))
manifests = sorted(input_root.rglob("audit_pasangan_e2.json"))
if archives:
    assert len(archives) == 1
    source = archives[0].parent
    with ZipFile(archives[0]) as archive:
        names = [Path(name) for name in archive.namelist()]
        assert all(name.parts[0] in {"640", "1280"} and not {"test", ".."}.intersection(name.parts) for name in names)
        archive.extractall(native)
else:
    assert len(manifests) == 1, "Lampirkan dataset privat native train/validation."
    source = manifests[0].parent
    tree = source / "native_train_val" if (source / "native_train_val/1280").is_dir() else source
    from concurrent.futures import ThreadPoolExecutor
    jobs = []
    for size in (640, 1280):
        for folder in ("images", "labels"):
            for split in ("train", "val"):
                dirname = Path(str(size), folder, split)
                source_folder = tree / dirname
                target_folder = native / dirname
                assert not (source_folder.parent / "test").exists()
                target_folder.mkdir(parents=True)
                jobs.extend((file, target_folder / file.name) for file in source_folder.iterdir() if file.is_file())
    assert len(jobs) == 6400
    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(lambda job: shutil.copyfile(*job), jobs))
    print("Native train and validation staged", flush=True)
audit_path = native / "audit_pasangan_e2.json"
shutil.copyfile(source / audit_path.name, audit_path)
audit = json.loads(audit_path.read_text())
assert audit["ready"] and audit["counts"] == {"paired": 1600}
weights = source / "yolo26n.pt"
assert sha256(weights) == WEIGHTS_SHA256
gt = project / "hasil_m1/coco_val_gt.json"
gt.parent.mkdir(parents=True)
shutil.copyfile(source / gt.name, gt)
ground_truth = json.loads(gt.read_text())
assert (len(ground_truth["images"]), len(ground_truth["annotations"])) == (356, 2205)
assert {entry["file_name"] for entry in ground_truth["images"]} == {
    p.name for p in (native / "1280/images/val").glob("*.jpg")
}
for split, count in (("train", 1244), ("val", 356)):
    known_ids = {Path(row["filename"]).stem for row in audit["files"] if row["split"] == split and row["status"] == "paired"}
    assert len(known_ids) == count
    for size in (640, 1280):
        for folder, extension in (("images", "jpg"), ("labels", "txt")):
            assert {p.stem for p in (native / f"{size}/{folder}/{split}").glob(f"*.{extension}")} == known_ids
def tree_sha256(root):
    result = digest()
    for folder in ("images/train", "images/val", "labels/train", "labels/val"):
        for file in sorted((root / folder).iterdir()):
            if file.suffix in {".jpg", ".txt"}:
                result.update(f"{file.relative_to(root).as_posix()} {sha256(file)}\n".encode("utf-8"))
    return result.hexdigest()

native_content_sha = {str(size): tree_sha256(native / str(size)) for size in (640, 1280)}
print("Native train/validation tree SHA-256:", native_content_sha)
sources = [
    "hasil_m2/evaluasi_e2.py", "hasil_m2/evaluasi_mosaic.py",
    "hasil_m2/evaluasi_e4_sahi.py", "hasil_m2/uji_tiling_640.py",
    "FP_Data_Mining_M1_Eksekusi_Lokal.ipynb",
]
source_hashes = {}
for name in sources:
    destination = project / name
    destination.parent.mkdir(exist_ok=True)
    urlretrieve(f"https://raw.githubusercontent.com/Jordyee/ciringoni-data-mining/{SOURCE_COMMIT}/Data_Mining_Ciringoni_M1/{name}", destination)
    source_hashes[name] = sha256(destination)
print("CPU", cpus, "| 1244 train, 356 val, 0 test | bobot awal", WEIGHTS_SHA256)

In [ ]:
from collections import Counter
from math import isclose
import random

SIZE = 1280
TILE = 640
OVERLAP = 0.20
VISIBILITY = 0.50
BATCH = 8
SEED = 42
stride = round(TILE * (1 - OVERLAP))
offsets = sorted(set(range(0, SIZE - TILE + 1, stride)) | {SIZE - TILE})

def clipped_box(box, ox, oy):
    c, x, y, w, h = box
    x0, x1 = (x - w / 2) * SIZE, (x + w / 2) * SIZE
    y0, y1 = (y - h / 2) * SIZE, (y + h / 2) * SIZE
    a, b, d, e = max(x0, ox), max(y0, oy), min(x1, ox + TILE), min(y1, oy + TILE)
    if d <= a or e <= b or (d - a) * (e - b) < VISIBILITY * (x1 - x0) * (y1 - y0):
        return None
    return int(c), ((a + d) / 2 - ox) / TILE, ((b + e) / 2 - oy) / TILE, (d - a) / TILE, (e - b) / TILE

assert all(isclose(got, expected) for got, expected in zip(
    clipped_box((0, 0.5, 0.3, 0.2, 0.2), 0, 0), (0, 0.9, 0.6, 0.2, 0.4), strict=True
))
assert clipped_box((0, 0.6, 0.3, 0.2, 0.2), 0, 0) is None
full_images = sorted((native / "1280/images/train").glob("*.jpg"))
positive, empty = [], []
original_boxes = 0
for image in full_images:
    label = native / "1280/labels/train" / f"{image.stem}.txt"
    rows = [list(map(float, line.split())) for line in label.read_text().splitlines()]
    original_boxes += len(rows)
    for y in offsets:
        for x in offsets:
            boxes = [clipped for row in rows if (clipped := clipped_box(row, x, y)) is not None]
            entry = (image, x, y, boxes)
            (positive if boxes else empty).append(entry)

rng = random.Random(SEED)
sources_with_positives = {image for image, *_ in positive}
assert sources_with_positives == set(full_images), "Setiap sumber train perlu punya minimal satu tile berobjek."
empty_count = min(len(empty), len(positive) // 3)
empty_count -= (len(positive) + empty_count) % BATCH
assert empty_count >= 0
chosen = positive + rng.sample(empty, empty_count)
assert len(chosen) % BATCH == 0
steps = len(chosen) // BATCH
tile_root = work / "tiles"
(tile_root / "images/train").mkdir(parents=True)
(tile_root / "labels/train").mkdir(parents=True)
tile_images = []
example_crops = []
by_source = {}
for image, x, y, boxes in chosen:
    by_source.setdefault(image, []).append((x, y, boxes))
for image, tile_specs in by_source.items():
    with Image.open(image) as source_image:
        assert source_image.size == (SIZE, SIZE)
        for x, y, boxes in tile_specs:
            stem = f"{image.stem}_x{x}_y{y}"
            image_path = tile_root / "images/train" / f"{stem}.jpg"
            crop = source_image.crop((x, y, x + TILE, y + TILE)).convert("RGB")
            crop.save(image_path, quality=95)
            target = tile_root / "labels/train" / f"{stem}.txt"
            target.write_text("".join(f"{c} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}\n" for c, cx, cy, w, h in boxes))
            tile_images.append(image_path)
            if boxes and len(example_crops) < 6:
                draw = ImageDraw.Draw(crop)
                for _, cx, cy, w, h in boxes:
                    draw.rectangle(((cx-w/2)*TILE, (cy-h/2)*TILE, (cx+w/2)*TILE, (cy+h/2)*TILE), outline="lime", width=3)
                example_crops.append(crop.resize((320, 320), Image.Resampling.LANCZOS))
assert len(set(tile_images)) == len(chosen)
examples = Image.new("RGB", (960, 640), "white")
for index, image in enumerate(example_crops):
    examples.paste(image, (320 * (index % 3), 320 * (index // 3)))
examples.save(output / "enam_tile_berlabel.png")
display(DisplayImage(filename=str(output / "enam_tile_berlabel.png")))
audit_tiles = {
    "source_images": len(full_images), "source_boxes": original_boxes,
    "tile_size": TILE, "overlap_nominal": OVERLAP, "tile_offsets_px": offsets,
    "min_visibility": VISIBILITY, "candidate_tiles": len(positive) + len(empty),
    "positive_tiles": len(positive), "empty_tiles": empty_count, "selected_tiles": len(chosen),
    "selected_boxes": sum(len(entry[3]) for entry in chosen), "batch": BATCH, "optimizer_steps_target": steps,
}
(output / "audit_tile_native.json").write_text(json.dumps(audit_tiles, indent=2) + "\n")
print(json.dumps(audit_tiles, indent=2))

In [ ]:
import time
from ultralytics import YOLO

val_images = native / "1280/images/val"
runs = {"E3_control_full_native": full_images * (len(chosen) // len(full_images)),
        "E3_tile_native_640": tile_images}
runs["E3_control_full_native"] += rng.sample(full_images, len(chosen) % len(full_images))
checkpoints = {}
training = {}
for name, images in runs.items():
    assert len(images) == BATCH * steps
    manifest = work / f"{name}.txt"
    manifest.write_text("".join(f"{image.as_posix()}\n" for image in images))
    yaml = work / f"{name}.yaml"
    yaml.write_text(json.dumps({"path": str(work), "train": str(manifest), "val": str(val_images), "names": ["infected"]}))
    count = Counter()
    def on_step(*_):
        count["optimizer_steps"] += 1
    def on_start(trainer):
        if len(trainer.train_loader) != steps or trainer.accumulate != 1:
            raise RuntimeError("Jumlah batch atau akumulasi gradient tidak sesuai anggaran.")
        trainer.optimizer.register_step_post_hook(on_step)
    def on_batch(trainer):
        count["minibatches"] += 1
    started = time.monotonic()
    model = YOLO(str(weights))
    model.add_callback("on_train_start", on_start)
    model.add_callback("on_train_batch_end", on_batch)
    model.train(
        data=str(yaml), project=str(output / "training"), name=name, epochs=1,
        imgsz=640, batch=BATCH, nbs=BATCH, device="cpu", workers=0,
        seed=SEED, deterministic=True, amp=False, mosaic=0.0,
        close_mosaic=0, optimizer="AdamW", lr0=0.002, lrf=1.0,
        warmup_epochs=0.0, val=False, plots=False, patience=0,
    )
    checkpoint = output / "training" / name / "weights/last.pt"
    assert checkpoint.is_file()
    assert dict(count) == {"minibatches": steps, "optimizer_steps": steps}, dict(count)
    run_info = {
        "checkpoint": str(checkpoint.relative_to(output)),
        "checkpoint_sha256": sha256(checkpoint),
        "source_images": len(full_images), "train_inputs": len(images), **dict(count),
        "training_seconds": time.monotonic() - started,
    }
    training[name] = run_info
    checkpoints[name] = checkpoint
    print(name, run_info)
    del model
(output / "training_e3.json").write_text(json.dumps(training, indent=2) + "\n")

In [ ]:
from importlib.metadata import version
import pandas as pd

evaluator_dir = project / "hasil_m2"
val_config = native / "1280/data.yaml"
val_config.write_text(json.dumps({"path": str(native / "1280"), "train": "images/train", "val": "images/val", "names": ["infected"]}))
for name, checkpoint in checkpoints.items():
    dest = output / "validation" / name
    command = [sys.executable, str(evaluator_dir / "evaluasi_e2.py"),
               str(checkpoint), str(val_config), str(audit_path), str(dest), "--size", "1280"]
    subprocess.run([*command, "--device", "cpu", "--batch", "4"], check=True)
    subprocess.run([*command, "--cpu-benchmark"], check=True)

sahi_destination = output / "validation/E3b_sahi"
sahi_command = [
    sys.executable, "-c", f"import evaluasi_e4_sahi as e; e.OVERLAP={OVERLAP!r}; e.main()",
    str(checkpoints["E3_tile_native_640"]), str(val_images), str(audit_path), str(sahi_destination),
    "--device", "cpu", "--evidence-reference", str(native / "640/images/val"),
]
subprocess.run([*sahi_command, "--smoke"], cwd=evaluator_dir, check=True)
subprocess.run(sahi_command, cwd=evaluator_dir, check=True)

rows = []
for name in checkpoints:
    result = output / "validation" / name
    row = json.loads((result / "metrik.json").read_text())
    predictions = result / "coco_val_predictions.json"
    assert row["gambar"] == 356 and row["imgsz"] == 1280
    assert 0 < row["cpu_median_ms"] <= row["cpu_p95_ms"]
    assert json.loads(predictions.read_text())
    rows.append(row | {"checkpoint": training[name]["checkpoint"], "predictions_sha256": sha256(predictions)})
table = pd.DataFrame(rows)
table.to_csv(output / "hasil_e3a.csv", index=False)
display(table[["id", "AP-small", "mAP50-95", "presisi_conf_0.05_iou_0.5", "recall_conf_0.05_iou_0.5", "cpu_median_ms", "cpu_p95_ms"]])
sahi_result = json.loads((sahi_destination / "metrik_e4_sahi.json").read_text())
assert sahi_result["checkpoint_sha256"] == training["E3_tile_native_640"]["checkpoint_sha256"]
assert (sahi_result["images"], sahi_result["gt_boxes"], sahi_result["overlap"]) == (356, 2205, OVERLAP)
sahi_table = pd.DataFrame([sahi_result[arm] | {"mode": arm} for arm in ("full", "sahi")])
sahi_table.to_csv(output / "hasil_e3b.csv", index=False)
display(sahi_table)
display(DisplayImage(filename=str(sahi_destination / "bukti_e4_sahi.png")))
proof = {
    "experiment": "E3 revised native tiling",
    "activity": ["CPU training", "CPU validation", "CPU latency with full frames", "CPU SAHI validation"],
    "source_commit": SOURCE_COMMIT, "source_sha256": source_hashes,
    "dataset_sha256": {file.name: sha256(file) for file in (audit_path, gt, *archives)},
    "native_tree_sha256": native_content_sha,
    "pretrained_sha256": WEIGHTS_SHA256,
    "version": {p: version(p) for p in ("ultralytics", "pycocotools", "sahi", "torch", "torchvision")},
    "python": sys.version.split()[0], "platform": platform.platform(), "cpu_affinity": cpus,
    "val_images": 356, "val_boxes": 2205, "training": training, "tiles": audit_tiles,
    "output_sha256": {str(file.relative_to(output)): sha256(file) for file in sorted(output.rglob("*"))
                      if file.suffix in {".json", ".csv", ".pt", ".png"}},
}
(output / "run_e3.json").write_text(json.dumps(proof, indent=2) + "\n")
print("Bukti E3 tersimpan pada", output)